# 🧹 Notebook 2 — Text Preprocessing
**Project**: LLM Response Preference Predictor  

**Canonical script:** `scripts/train.py` (text parsing/cleaning in Stages **2–3**). Align these notebook cells with that file when iterating on the pipeline.

---
### What this notebook covers
- Why we clean LLM text (markdown, URLs, special chars)
- Step-by-step cleaning pipeline with before/after examples
- Building combined text fields (full context vs vocabulary diff)
- Saving cleaned data for the feature engineering notebook

In [1]:
import pandas as pd
import numpy as np
import json, ast, re, os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../data', exist_ok=True)
print('Libraries loaded ✓')

Libraries loaded ✓


In [2]:
TRAIN_CSV = '../data/train.csv'
df = pd.read_csv(TRAIN_CSV)

def parse_field(val):
    if pd.isna(val): return ''
    v = str(val).strip()
    if v.startswith('['):
        try: return ' '.join(str(p) for p in json.loads(v))
        except:
            try: return ' '.join(str(p) for p in ast.literal_eval(v))
            except: return v
    return v

df['prompt_text']     = df['prompt'].apply(parse_field)
df['response_a_text'] = df['response_a'].apply(parse_field)
df['response_b_text'] = df['response_b'].apply(parse_field)

def get_label(row):
    if row['winner_model_a'] == 1: return 0
    elif row['winner_model_b'] == 1: return 1
    else: return 2
df['label'] = df.apply(get_label, axis=1)

print(f'Loaded {len(df):,} rows')

Loaded 57,477 rows


## 2.1 Why Do We Need to Clean Text?

LLM responses contain formatting artifacts that add tokens without meaning:

| Artifact | Example | Problem |
|---|---|---|
| Code blocks | ` ```python\nprint(x)\n``` ` | Pollutes vocabulary with syntax |
| Markdown bold | `**important**` | Creates `important` as 2 tokens |
| URLs | `https://example.com` | Unique per document, useless |
| Numbers | `42`, `3.14` | Usually formatting noise |
| Stopwords | `the`, `a`, `is` | Appear everywhere, no signal |

In [3]:
STOPWORDS = {
    'a','an','the','and','or','but','in','on','at','to','for','of','with',
    'by','from','is','are','was','were','be','been','has','have','had',
    'do','does','did','will','would','could','should','may','might','shall',
    'this','that','these','those','i','you','he','she','it','we','they',
    'me','him','her','us','them','my','your','his','its','our','their',
    'what','which','who','how','when','where','why','not','no','so','as',
    'up','out','if','about','into','than','then','some','more','also',
    'just','like','can','all','there','been','being','very','get',
}

def clean_text(text):
    """
    Full cleaning pipeline for LLM response text.
    Returns cleaned, tokenised, stopword-free string.
    """
    if not isinstance(text, str) or text.strip() == '':
        return ''
    # Step 1: lowercase
    text = text.lower()
    # Step 2: remove code blocks (multi-line ```...```)
    text = re.sub(r'```[\s\S]{0,3000}?```', ' ', text)
    # Step 3: remove inline code (`...`)
    text = re.sub(r'`[^`]+`', ' ', text)
    # Step 4: strip markdown bold/italic
    text = re.sub(r'\*{1,3}([^*]{0,300})\*{1,3}', r'\1', text)
    # Step 5: strip markdown headers
    text = re.sub(r'#{1,6}\s', ' ', text)
    # Step 6: strip markdown links [text](url) → text
    text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)
    # Step 7: remove URLs
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    # Step 8: keep only alphabetic chars
    text = re.sub(r'[^a-z\s]', ' ', text)
    # Step 9: collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # Step 10: remove stopwords and very short tokens
    tokens = [w for w in text.split() if w not in STOPWORDS and len(w) >= 2]
    return ' '.join(tokens)

print('clean_text() function defined ✓')

clean_text() function defined ✓


## 2.2 Before / After Examples

In [4]:
examples = [
    "**Important**: Visit https://example.com for more info. Use `print('hello')` to test.",
    "```python\nimport pandas as pd\ndf = pd.read_csv('data.csv')\n```\nThis code loads a CSV file.",
    "## Step 1: Install\nRun the following command:\n- Install Python 3.10\n- Install dependencies",
]

print('BEFORE → AFTER cleaning examples:')
print('='*70)
for i, ex in enumerate(examples, 1):
    cleaned = clean_text(ex)
    print(f'\nExample {i}:')
    print(f'  BEFORE: {ex[:100]}')
    print(f'  AFTER : {cleaned[:100]}')

BEFORE → AFTER cleaning examples:

Example 1:
  BEFORE: **Important**: Visit https://example.com for more info. Use `print('hello')` to test.
  AFTER : important visit info use test

Example 2:
  BEFORE: ```python
import pandas as pd
df = pd.read_csv('data.csv')
```
This code loads a CSV file.
  AFTER : code loads csv file

Example 3:
  BEFORE: ## Step 1: Install
Run the following command:
- Install Python 3.10
- Install dependencies
  AFTER : step install run following command install python install dependencies


In [5]:
# Real data example
idx = 0
print('REAL DATA EXAMPLE (row 0):')
print('='*70)
print(f'BEFORE ({len(df["response_a_text"].iloc[idx].split())} words):')
print(df['response_a_text'].iloc[idx][:300])
cleaned = clean_text(df['response_a_text'].iloc[idx])
print(f'\nAFTER ({len(cleaned.split())} words):')
print(cleaned[:300])

REAL DATA EXAMPLE (row 0):
BEFORE (674 words):
The question of whether it is morally right to aim for a certain percentage of females in managerial positions is a complex ethical issue that involves considerations of fairness, equality, diversity, and discrimination.

Here are some arguments in favor of and against such policies:

**Arguments in

AFTER (411 words):
question whether morally right aim certain percentage females managerial positions complex ethical issue involves considerations fairness equality diversity discrimination here arguments favor against such policies arguments favor correcting historical inequities women historically underrepresented 


## 2.3 Apply Cleaning to Full Dataset

In [6]:
import time
t0 = time.time()

print('Cleaning all text columns...')
df['prompt_clean']     = df['prompt_text'].apply(clean_text)
df['response_a_clean'] = df['response_a_text'].apply(clean_text)
df['response_b_clean'] = df['response_b_text'].apply(clean_text)

print(f'Done in {time.time()-t0:.0f}s')
print(f'\nAverage word count after cleaning:')
print(f'  prompt_clean    : {df["prompt_clean"].apply(lambda x: len(x.split())).mean():.0f} words')
print(f'  response_a_clean: {df["response_a_clean"].apply(lambda x: len(x.split())).mean():.0f} words')
print(f'  response_b_clean: {df["response_b_clean"].apply(lambda x: len(x.split())).mean():.0f} words')

Cleaning all text columns...
Done in 19s

Average word count after cleaning:
  prompt_clean    : 32 words
  response_a_clean: 115 words
  response_b_clean: 116 words


## 2.4 Build Combined Text Fields

We create **two** combined representations:

| Field | Content | Why |
|---|---|---|
| `text_full` | PROMPT + RESPONSE_A + RESPONSE_B | Gives TF-IDF the full context |
| `text_diff` | Words unique to A + words unique to B | Captures what makes each response DIFFERENT |

In [7]:
# text_full: Full context concatenation
df['text_full'] = (
    'PROMPT: '     + df['prompt_clean'] + ' ' +
    'RESPONSE_A: ' + df['response_a_clean'] + ' ' +
    'RESPONSE_B: ' + df['response_b_clean']
)

# text_diff: Symmetric vocabulary difference
# Words only in A (not B) + words only in B (not A)
# This is the DISCRIMINATIVE signal — what makes each response unique
def get_diff(ta, tb):
    sa, sb = set(ta.split()), set(tb.split())
    return ' '.join(sorted(sa - sb)) + ' ' + ' '.join(sorted(sb - sa))

df['text_diff'] = [
    get_diff(a, b)
    for a, b in zip(df['response_a_clean'], df['response_b_clean'])
]

print('Combined fields created:')
print(f'  text_full avg length: {df["text_full"].str.len().mean():.0f} chars')
print(f'  text_diff avg length: {df["text_diff"].str.len().mean():.0f} chars')
print(f'\nSample text_diff (shows UNIQUE vocabulary):')
print(df['text_diff'].iloc[0][:200])

Combined fields created:
  text_full avg length: 2007 chars
  text_diff avg length: 777 chars

Sample text_diff (shows UNIQUE vocabulary):
absolutely across address advance against age ah ahead aim aiming aloha among ancestors anchovies app appointed approach aspects atop bacon balance barriers battle beach belong bias blind bon broader 


In [8]:
# Save cleaned data
cols_to_save = ['id', 'model_a', 'model_b', 'label',
                'prompt_clean', 'response_a_clean', 'response_b_clean',
                'text_full', 'text_diff',
                'prompt_text', 'response_a_text', 'response_b_text']
df[cols_to_save].to_csv('../data/df_processed.csv', index=False)
print('Saved → ../data/df_processed.csv')
print(f'Rows: {len(df):,}  Columns: {len(cols_to_save)}')

Saved → ../data/df_processed.csv
Rows: 57,477  Columns: 12


## 2.5 Summary

✅ Parsed multi-turn JSON conversation fields  
✅ Lowercased, removed markdown/URLs/code blocks/numbers  
✅ Filtered stopwords and short tokens  
✅ Created `text_full` (context) and `text_diff` (discriminative)  
✅ Saved cleaned dataset to `data/df_processed.csv`  

**Next step → Notebook 03: Feature Engineering**